Part 1 Risk engine

-pip install downloads libraries and makes them available to import.

-xgboost = powerful tree boosting models.

-shap = explanation tool; TreeExplainer is optimized for tree models.

-scikit-learn = preprocessing, metrics, training utilities.

-joblib = saving models to disk.

-fastapi/uvicorn = serving the model as an API (deployment).

-import X as Y loads a library and optionally renames it (xgboost as xgb is conventional).

-Pipeline lets you chain preprocessing → model so you don’t forget steps.

-ColumnTransformer applies different preprocessing to numeric vs categorical columns.

The FEATURES list is your input schema—like an API contract.

Separating numeric vs categorical lets us preprocess correctly:

numeric → fill missing with median

categorical → one-hot encode

imports and feature list

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import mean_absolute_error

import xgboost as xgb
import shap
import joblib
import uuid


In [16]:
# These are the EXACT fields a booking request must contain.
FEATURES = [
    "driver_age", "account_age_days", "completed_trips", "prior_claims", "rating",
    "cancellations_90d", "trip_days", "lead_time_hours", "pickup_hour", "weekend",
    "vehicle_class", "market", "high_theft_area"
]

NUM_FEATURES = [
    "driver_age", "account_age_days", "completed_trips", "prior_claims", "rating",
    "cancellations_90d", "trip_days", "lead_time_hours", "pickup_hour", "weekend",
    "high_theft_area"
]

CAT_FEATURES = ["vehicle_class", "market"]


These features represent what a marketplace knows at checkout:

renter history (account age, trips, claims)

trip context (lead time, pickup time, weekend)

car type (vehicle class)

city/market

2. Data: generate a synthetic marketplace dataset (claim + cost)
-Real Turo is private so i need to generate simulations inorder to accumliate usable data that can mimic realistic:

driver history

trip context/details

car type

market location

outcomes (claim yes/No, claim cost)

In [3]:
def make_synthetic_marketplace_data(n=80_000, seed=7):
    rng = np.random.default_rng(seed)

    # -------------------------
    # Driver features (known at booking time)
    driver_age = rng.integers(21, 70, size=n)
    account_age_days = rng.integers(0, 2000, size=n)      # days since signup
    completed_trips = rng.poisson(lam=5, size=n)          # prior completed rentals
    prior_claims = rng.poisson(lam=0.15, size=n)          # prior damage claims
    rating = np.clip(rng.normal(4.85, 0.15, size=n), 1.0, 5.0)
    cancellations_90d = rng.poisson(lam=0.3, size=n)

    # -------------------------
    # Trip context (known at booking time)
    trip_days = rng.integers(1, 15, size=n)
    lead_time_hours = np.clip(rng.normal(36, 24, size=n), 0, 240)  # last-minute vs planned
    pickup_hour = rng.integers(0, 24, size=n)
    weekend = (rng.integers(0, 7, size=n) >= 5).astype(int)

    # -------------------------
    # Vehicle + market (known at booking time)
    vehicle_class = rng.choice(
        ["economy", "standard", "premium", "performance", "suv"],
        size=n, p=[0.35, 0.35, 0.15, 0.05, 0.10]
    )
    market = rng.choice(["LA", "SF", "SD", "LV", "PHX"], size=n,
                        p=[0.28, 0.20, 0.20, 0.12, 0.20])

    # Example proxy feature: higher theft-risk context
    high_theft_area = ((market == "LA") & (pickup_hour >= 20)).astype(int)

    # -------------------------
    # Label 1: Probability of claim (synthetic ground truth)
    # "z" is log-odds: higher z -> higher probability
    base = -3.7  # controls overall claim rate
    z = (
        base
        + 0.8 * (prior_claims > 0)
        + 0.35 * (cancellations_90d > 0)
        + 0.28 * (lead_time_hours < 6)
        + 0.20 * weekend
        + 0.40 * high_theft_area
        + 0.15 * (trip_days >= 7)
        - 0.0006 * account_age_days
        - 0.04 * (rating - 4.7)
        + 0.25 * (vehicle_class == "premium")
        + 0.55 * (vehicle_class == "performance")
        + 0.12 * (market == "LV")
    )
    p_claim = 1 / (1 + np.exp(-z))
    claim = rng.binomial(1, p_claim, size=n)

    # -------------------------
    # Label 2: Claim severity (cost) if claim happens
    # Costs are skewed: many small, some huge -> lognormal noise
    base_cost = 900
    multiplier = (
        1.0
        + 0.35 * (vehicle_class == "premium")
        + 1.00 * (vehicle_class == "performance")
        + 0.25 * (vehicle_class == "suv")
        + 0.15 * (trip_days >= 7)
        + 0.35 * high_theft_area
        + 0.25 * (prior_claims > 0)
    )
    noise = rng.lognormal(mean=0.0, sigma=0.60, size=n)
    claim_cost = claim * (base_cost * multiplier * noise)

    df = pd.DataFrame({
        "driver_age": driver_age,
        "account_age_days": account_age_days,
        "completed_trips": completed_trips,
        "prior_claims": prior_claims,
        "rating": rating,
        "cancellations_90d": cancellations_90d,
        "trip_days": trip_days,
        "lead_time_hours": lead_time_hours,
        "pickup_hour": pickup_hour,
        "weekend": weekend,
        "vehicle_class": vehicle_class,
        "market": market,
        "high_theft_area": high_theft_area,
        "claim": claim,
        "claim_cost": claim_cost
    })
    return df

df = make_synthetic_marketplace_data()
df[["claim", "claim_cost"]].describe()
df.head(50)


,driver_age,account_age_days,completed_trips,prior_claims,rating,cancellations_90d,trip_days,lead_time_hours,pickup_hour,weekend,vehicle_class,market,high_theft_area,claim,claim_cost
0,67,1189,4,0,4.760206,0,9,24.225158,16,0,premium,SF,0,0,0.0
1,51,126,3,0,4.843361,1,1,0.000000,9,1,premium,SF,0,0,0.0
2,54,0,3,0,4.789813,2,12,18.613872,16,0,economy,LA,0,0,0.0
3,64,770,4,0,4.926871,0,1,13.254087,10,0,performance,SF,0,0,0.0
4,49,423,3,0,4.733791,1,4,10.313462,23,1,economy,LA,1,0,0.0
5,59,1624,2,0,4.875291,0,7,27.519837,16,0,standard,LA,0,0,0.0
6,61,1324,7,1,4.952052,0,12,28.586758,11,0,standard,SD,0,0,0.0
7,32,1858,8,1,4.988701,0,7,72.726953,17,0,economy,LV,0,0,0.0
8,23,1736,5,0,4.867830,0,3,68.368895,21,0,standard,LA,1,0,0.0
9,35,1658,6,0,4.821416,0,9,56.831672,5,0,premium,LA,0,0,0.0


def ... defines a reusable function.

rng = np.random.default_rng(seed) makes randomness reproducible (important for experiments).

pd.DataFrame({...}) builds a table like a spreadsheet.

3. Feature setup + train/validation split (time-like discipline)

FEATURES is a list of column names.

stratify=y_claim keeps claim rate consistent between train and val (important for rare events).

Im training the model on past historical trips and validating on new trips

In [4]:
FEATURES = [
    "driver_age","account_age_days","completed_trips","prior_claims","rating",
    "cancellations_90d","trip_days","lead_time_hours","pickup_hour","weekend",
    "vehicle_class","market","high_theft_area"
]

X = df[FEATURES]
y_claim = df["claim"]
y_cost = df["claim_cost"]

X_train, X_val, y_claim_train, y_claim_val = train_test_split(
    X, y_claim, test_size=0.2, random_state=42, stratify=y_claim
)


4) Preprocessing for tree models (OneHot + numeric impute)

-ColumnTransformer applies different rules to different columns.

-Numeric pipeline:

fills missing numbers with the median (robust to outliers).

-Categorical pipeline:

fills missing category with most common

-one-hot encodes categories into 0/1 columns.

In [5]:
num_features = [
    "driver_age","account_age_days","completed_trips","prior_claims","rating",
    "cancellations_90d","trip_days","lead_time_hours","pickup_hour","weekend","high_theft_area"
]
cat_features = ["vehicle_class","market"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ]
)


If a renter’s profile is missing a field (common in real systems), we can still score the booking.

5) Model A: Claim probability using XGBoost + calibration

I used XGBoost because the many small decision trees helps capture non-linear patters that will occur like performance cars at night being disproportionately more risky

5A) Base XGBoost classifier pipeline

In [6]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",   # faster training
    random_state=42
)

claim_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", xgb_clf)
])

claim_pipeline.fit(X_train, y_claim_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

XGBClassifier(...) creates the model object (hyperparameters inside).

Pipeline([...]) chains preprocessing + model.

.fit(...) trains the model.

ML

n_estimators: how many trees.

max_depth: how complex each tree is.

learning_rate: how strongly each new tree changes the model.

subsample/colsample_bytree: randomness to reduce overfitting.

This is the “will this booking likely turn into a claim?” model

5B) Calibrate probability outputs:

Tree models are good at ranking but not always calibrated properly

In [7]:
claim_model_calibrated = CalibratedClassifierCV(
    estimator=claim_pipeline,
    method="isotonic",
    cv=3
)
claim_model_calibrated.fit(X_train, y_claim_train)

p_val = claim_model_calibrated.predict_proba(X_val)[:, 1]
print("AUC:", roc_auc_score(y_claim_val, p_val))
print("AP :", average_precision_score(y_claim_val, p_val))
print("Brier:", brier_score_loss(y_claim_val, p_val))


AUC: 0.619385164070004
AP : 0.04065986157106888
Brier: 0.022590889315947764


-CalibratedClassifierCV wraps a model and learns a mapping from raw scores → better probabilities.

-cv=3 means it uses cross-validation splits for calibration.

This is critical if you will later say:

“deposit threshold is p_claim >= 6%”

because you want 6% to actually mean something.

6) Model B: Claim severity (cost) using XGBoost regressor

Im training severity only on trips with (claim==1 because only those have meaningful cost)

In [8]:
df_claims = df[df["claim"] == 1].copy()

Xc = df_claims[FEATURES]
yc = df_claims["claim_cost"]

Xc_train, Xc_val, yc_train, yc_val = train_test_split(
    Xc, yc, test_size=0.2, random_state=42
)

# log-transform to stabilize skew
yc_train_log = np.log1p(yc_train)
yc_val_log = np.log1p(yc_val)

xgb_reg = xgb.XGBRegressor(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

severity_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", xgb_reg)
])

severity_pipeline.fit(Xc_train, yc_train_log)

pred_log = severity_pipeline.predict(Xc_val)
pred_cost = np.expm1(pred_log)
true_cost = np.expm1(yc_val_log)

print("Severity MAE ($):", round(mean_absolute_error(true_cost, pred_cost), 2))


Severity MAE ($): 681.19


-df[df["claim"] == 1] filters rows where claim occurred.

-np.log1p(cost) means log(1 + cost).adding 1 prevents log(0) problems.

-Log transform makes learning easier since some cost cand be very large

- This estimates if something goes wrong roughly how expensice will it be

7) Expected loss function (underwriting core)

p_claim is a fraction like 0.08 (8%).

exp_cost_if_claim might be $2,100.

expected loss = 0.08 × 2100 = $168.

Expected loss is what insurance-style decisioning systems optimize.

In [9]:
def predict_expected_loss(request_df: pd.DataFrame):
    """
    request_df: DataFrame with one or many rows, FEATURES columns.
    Returns:
      p_claim (probability)
      exp_cost_if_claim (expected dollars if claim happens)
      expected_loss (probability * dollars)
    """
    p_claim = claim_model_calibrated.predict_proba(request_df)[:, 1]

    # severity outputs log(cost); convert back to dollars
    exp_log_cost = severity_pipeline.predict(request_df)
    exp_cost_if_claim = np.expm1(exp_log_cost)

    expected_loss = p_claim * exp_cost_if_claim
    return p_claim, exp_cost_if_claim, expected_loss


8) Policy engine (contract rules) — now driven by expected loss

-Deposit + telematics = “you can rent, but with extra safeguards”

-Manual review = “trust & safety check”

-Decline = “risk too high”

-@dataclass creates a simple container object for readability.

In [10]:
from dataclasses import dataclass

@dataclass
class PolicyDecision:
    decision: str
    recommended_terms: dict
    reasons: list

def policy_engine(p_claim: float, expected_loss: float, request: dict) -> PolicyDecision:
    terms = {
        "deposit_required": False,
        "in_person_checkin": False,
        "telematics_opt_in": False,
        "max_trip_days": None
    }

    # Example thresholds (you tune these in experiments)
    if p_claim >= 0.25 or expected_loss >= 350:
        decision = "DECLINE"
    elif p_claim >= 0.12 or expected_loss >= 200:
        decision = "REVIEW"
    elif p_claim >= 0.06 or expected_loss >= 120:
        decision = "APPROVE_WITH_CONDITIONS"
    else:
        decision = "APPROVE"

    # Safeguards
    if decision in ["APPROVE_WITH_CONDITIONS", "REVIEW"]:
        terms["deposit_required"] = True
        terms["telematics_opt_in"] = True
    if decision == "APPROVE_WITH_CONDITIONS":
        terms["in_person_checkin"] = True

    # Vehicle-specific tightening (performance cars)
    if request.get("vehicle_class") == "performance" and decision != "DECLINE":
        terms["deposit_required"] = True
        terms["telematics_opt_in"] = True
        terms["max_trip_days"] = 3
        if decision == "APPROVE":
            decision = "APPROVE_WITH_CONDITIONS"

    reasons = []  # SHAP will provide “true” reasons soon
    return PolicyDecision(decision=decision, recommended_terms=terms, reasons=reasons)


9) SHAP with TreeExplainer (fast + clean)

SHAP answers:

“For THIS booking, which inputs pushed the predicted risk UP or DOWN compared to an average booking?”

With XGBoost, shap.TreeExplainer is optimized and fast.

9B) Extract transformed feature names (for readable explanations)

In [11]:
def get_transformed_feature_names(preprocessor, numeric_features, categorical_features):
    num_names = numeric_features

    cat_transformer = preprocessor.named_transformers_["cat"]
    ohe = cat_transformer.named_steps["onehot"]
    cat_names = ohe.get_feature_names_out(categorical_features).tolist()

    return num_names + cat_names

preprocessor = claim_pipeline.named_steps["preprocess"]
feature_names = get_transformed_feature_names(preprocessor, num_features, cat_features)
len(feature_names), feature_names[:12]


(21,
 ['driver_age',
  'account_age_days',
  'completed_trips',
  'prior_claims',
  'rating',
  'cancellations_90d',
  'trip_days',
  'lead_time_hours',
  'pickup_hour',
  'weekend',
  'high_theft_area',
  'vehicle_class_economy'])

This pulls the OneHotEncoder out of the nested pipeline.

Ask it for the expanded column names.

This ensures we can say “market is LA” instead of “column_184”.

9C) Build TreeExplainer for the XGBoost claim model

.sample(2000) takes a small subset to use as a background distribution.

TreeExplainer(xgb_model) prepares SHAP for your tree model.

SHAP needs a sense of “typical data” to define what “baseline” means. Baseline is like:

“average booking risk before we know anything specific about this booking.”

In [12]:
# Transform a background dataset (used to anchor SHAP expectations)
X_bg = preprocessor.transform(X_train.sample(2000, random_state=42))

# Extract the trained XGBoost model
xgb_model = claim_pipeline.named_steps["model"]

# TreeExplainer: fast for tree models
tree_explainer = shap.TreeExplainer(xgb_model)

# SHAP baseline (expected value) is internal to the explainer


9D) Explain a single booking request with SHAP (top factors)

A positive SHAP value means that feature pushed the model toward “higher claim risk.”

A negative SHAP value means it pushed risk lower.

The magnitude tells you how strong that push was.

This provides explanations that the average user can read and understand:

“Vehicle class is performance” pushed risk up.

“Last-minute booking” pushed risk up.

“High driver rating” pushed risk down.

In [13]:
def humanize_feature(feat: str):
    if feat.startswith("market_"):
        return f"Market is {feat.replace('market_', '')}"
    if feat.startswith("vehicle_class_"):
        return f"Vehicle class is {feat.replace('vehicle_class_', '')}"
    mapping = {
        "lead_time_hours": "Last-minute booking (low lead time)",
        "account_age_days": "Newer account (low account age)",
        "completed_trips": "Limited trip history",
        "prior_claims": "Prior claims history",
        "rating": "Driver rating",
        "trip_days": "Longer trip duration",
        "pickup_hour": "Late-night/early pickup time",
        "weekend": "Weekend booking",
        "high_theft_area": "Higher-risk pickup area/time",
        "cancellations_90d": "Recent cancellations"
    }
    return mapping.get(feat, feat)

def shap_explain_request(request: dict, top_k: int = 6):
    # 1) raw row
    x_raw = pd.DataFrame([request])[FEATURES]

    # 2) transform to model input
    x_trans = preprocessor.transform(x_raw)

    # 3) SHAP values for this row
    # For binary classification, shap_values shape: (1, num_features)
    shap_vals = tree_explainer.shap_values(x_trans)

    # 4) flatten
    shap_vec = np.array(shap_vals).reshape(-1)

    # 5) pair with names
    pairs = list(zip(feature_names, shap_vec))

    # 6) sort by strongest absolute effect
    pairs_sorted = sorted(pairs, key=lambda t: abs(t[1]), reverse=True)

    # 7) split up/down
    up = [(humanize_feature(f), float(v)) for f, v in pairs_sorted if v > 0][:top_k]
    down = [(humanize_feature(f), float(v)) for f, v in pairs_sorted if v < 0][:top_k]

    return {"top_risk_increasing": up, "top_risk_decreasing": down}


10) End-to-end scoring (calibrated probability + expected loss + SHAP reasons)

model will score the booking request at checkout. The system outputs probability of claim, expected cost if a claim happens, and expected loss. Then the policy engine chooses approve/review/decline. Finally, SHAP provides the top factors that drove the risk prediction so ops teams can understand and trust the decision.”

In [14]:
import uuid

def score_booking_request_tree_v4(request: dict):
    request_id = str(uuid.uuid4())
    req_df = pd.DataFrame([request])[FEATURES]

    # Underwriting numbers
    p_claim, exp_cost, exp_loss = predict_expected_loss(req_df)
    p_claim = float(p_claim[0])
    exp_cost = float(exp_cost[0])
    exp_loss = float(exp_loss[0])

    # Policy decision
    policy = policy_engine(p_claim, exp_loss, request)

    # SHAP explanations (compute when needed)
    # In real production, you might only compute if decision != APPROVE
    shap_reasons = shap_explain_request(request, top_k=6)

    result = {
        "request_id": request_id,
        "p_claim": p_claim,
        "expected_cost_if_claim": exp_cost,
        "expected_loss": exp_loss,
        "policy": {
            "decision": policy.decision,
            "recommended_terms": policy.recommended_terms,
            "reasons": shap_reasons  # attach SHAP “why”
        }
    }
    return result

example_request = {
    "driver_age": 26,
    "account_age_days": 12,
    "completed_trips": 0,
    "prior_claims": 0,
    "rating": 4.9,
    "cancellations_90d": 1,
    "trip_days": 3,
    "lead_time_hours": 2,
    "pickup_hour": 22,
    "weekend": 1,
    "vehicle_class": "performance",
    "market": "LA",
    "high_theft_area": 1
}

score_booking_request_tree_v4(example_request)


{'request_id': '494a7cca-e5b5-4568-8666-79c251b9be0b',
 'p_claim': 0.03714486211538315,
 'expected_cost_if_claim': 1605.8797607421875,
 'expected_loss': 59.650182286653035,
 'policy': {'decision': 'APPROVE_WITH_CONDITIONS',
  'recommended_terms': {'deposit_required': True,
   'in_person_checkin': False,
   'telematics_opt_in': True,
   'max_trip_days': 3},
  'reasons': {'top_risk_increasing': [('Newer account (low account age)',
     0.8001431822776794),
    ('Recent cancellations', 0.23890703916549683),
    ('Vehicle class is performance', 0.23745259642601013),
    ('Higher-risk pickup area/time', 0.22187690436840057),
    ('Late-night/early pickup time', 0.09179392457008362),
    ('Weekend booking', 0.0389658659696579)],
   'top_risk_decreasing': [('Limited trip history', -0.38210320472717285),
    ('Prior claims history', -0.15605129301548004),
    ('Longer trip duration', -0.08760061115026474),
    ('driver_age', -0.07926491647958755),
    ('Last-minute booking (low lead time)', -0